# Histogramas e tabelas comparativas do escalonamento (Anexo A.1)

Este notebook gera as figuras da seção **A.1.5** e as duas tabelas da seção **A.1.6** do documento.
Ele é o par do `escalonamento_anexo_a1.ipynb`, que trata das seções A.1.2 a A.1.4: as equações e as
constantes vêm de lá e são apenas reaplicadas aqui, para que as figuras e as tabelas saiam dos mesmos
números publicados.

**Base de entrada.** Lê `data/processed/base_analitica.parquet`, gerado por
`notebooks/pre-processamento.ipynb`. A pasta `data/` não é versionada, por compromisso entre o Inteli
e o parceiro, então o arquivo precisa existir localmente antes da execução. Se ele não for encontrado,
a base é reconstruída a partir de `data/raw` com `scripts/preprocessamento_nps.py`. A contagem de
484.915 registros é verificada por `assert`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = next(
    (c for c in [Path('.'), Path('..'), Path('/content')] if (c / 'data').exists() or (c / 'scripts').exists()),
    Path('.'),
)

caminho_base = RAIZ / 'data' / 'processed' / 'base_analitica.parquet'

if caminho_base.exists():
    df = pd.read_parquet(caminho_base)
else:
    sys.path.insert(0, str((RAIZ / 'scripts').resolve()))
    from preprocessamento_nps import integrar_bases, preparar_base_analitica

    df_bruto, _ = integrar_bases(RAIZ / 'data' / 'raw')
    df = preparar_base_analitica(df_bruto)

assert len(df) == 484915, f'a base analítica deveria ter 484.915 registros e tem {len(df)}'

PASTA_ASSETS = RAIZ / 'assets'
PASTA_ASSETS.mkdir(exist_ok=True)

print('Registros na base analítica:', f'{len(df):,}'.replace(',', '.'))

Registros na base analítica: 484.915


## 1. Reaplicação das equações da seção A.1.4

As mesmas constantes publicadas no documento, com as quatro casas decimais. A conferência de que elas
reproduzem a coluna escalonada está na seção 6 do `escalonamento_anexo_a1.ipynb` e não é repetida aqui.

In [2]:
MEDIA_TEMPO_VOO, DESVIO_TEMPO_VOO = 207.1565, 208.2920
MEDIA_ATRASO_CHEGADA, DESVIO_ATRASO_CHEGADA = 25.6551, 136.4553
MINIMO_QTDE_VIAGENS_12M, MAXIMO_QTDE_VIAGENS_12M = 0, 107

df['TEMPO_VOO_ESC'] = (df['TEMPO_VOO'] - MEDIA_TEMPO_VOO) / DESVIO_TEMPO_VOO
df['ATRASO_CHEGADA_ESC'] = (df['ATRASO_CHEGADA'] - MEDIA_ATRASO_CHEGADA) / DESVIO_ATRASO_CHEGADA
df['QTDE_VIAGENS_12M_ESC'] = (
    (df['QTDE_VIAGENS_12M'] - MINIMO_QTDE_VIAGENS_12M)
    / (MAXIMO_QTDE_VIAGENS_12M - MINIMO_QTDE_VIAGENS_12M)
)

df[['TEMPO_VOO_ESC', 'ATRASO_CHEGADA_ESC', 'QTDE_VIAGENS_12M_ESC']].describe().round(4)

,TEMPO_VOO_ESC,ATRASO_CHEGADA_ESC,QTDE_VIAGENS_12M_ESC
count,484675.0000,484915.0000,484760.0000
mean,-0.0000,0.0000,0.0296
std,1.0000,1.0000,0.0506
min,-0.8265,-0.1880,0.0000
25%,-0.6105,-0.1880,0.0000
50%,-0.3704,-0.1880,0.0093
75%,0.2297,-0.1880,0.0374
max,19.7456,31.4634,1.0000


## 2. Histogramas antes e depois (figuras 1 a 6 da seção A.1.5)

Os seis gráficos usam a mesma quantidade de classes e o mesmo estilo, porque o ponto da comparação é
que só o eixo horizontal muda. Cada arquivo é gravado em `assets/` com o nome citado no documento.

In [3]:
COR = '#1f5c99'
CLASSES = 60

FIGURAS = [
    ('hist_tempo_voo_original', 'TEMPO_VOO', 'TEMPO_VOO antes do escalonamento', 'Duração do deslocamento (minutos)'),
    ('hist_tempo_voo_padronizado', 'TEMPO_VOO_ESC', 'TEMPO_VOO após padronização', 'Escore z'),
    ('hist_atraso_chegada_original', 'ATRASO_CHEGADA', 'ATRASO_CHEGADA antes do escalonamento', 'Atraso na chegada (minutos)'),
    ('hist_atraso_chegada_padronizado', 'ATRASO_CHEGADA_ESC', 'ATRASO_CHEGADA após padronização', 'Escore z'),
    ('hist_qtde_viagens_original', 'QTDE_VIAGENS_12M', 'QTDE_VIAGENS_12M antes do escalonamento', 'Viagens em 12 meses'),
    ('hist_qtde_viagens_normalizado', 'QTDE_VIAGENS_12M_ESC', 'QTDE_VIAGENS_12M após normalização', 'Valor normalizado (0 a 1)'),
]

for arquivo, coluna, titulo, rotulo_x in FIGURAS:
    figura, eixo = plt.subplots(figsize=(9, 5.4), dpi=100)
    eixo.hist(df[coluna].dropna(), bins=CLASSES, color=COR, edgecolor='white', linewidth=0.5)
    eixo.set_title(titulo, fontsize=13, pad=14)
    eixo.set_xlabel(rotulo_x)
    eixo.set_ylabel('Frequência')
    eixo.grid(axis='y', color='#dddddd', linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ('top', 'right'):
        eixo.spines[lado].set_visible(False)
    figura.tight_layout()
    figura.savefig(PASTA_ASSETS / f'{arquivo}.png')
    plt.close(figura)

print(f'{len(FIGURAS)} figuras gravadas em {PASTA_ASSETS}')

6 figuras gravadas em ../assets


## 3. Tabelas comparativas dos dez primeiros registros (seção A.1.6)

Os registros são identificados pelo `RESPONDENT_ID`, chave de integração das quatro fontes e único na
base, e não pela posição: a posição depende da ordem em que `pre-processamento.ipynb` concatena as
fontes, e uma mudança nessa ordem trocaria as linhas da tabela publicada sem aviso.

São os dez primeiros registros da base analítica, sem seleção de valores, para que a amostra não pareça
escolhida a favor do argumento.

In [4]:
VARIAVEIS = ['TEMPO_VOO', 'ATRASO_CHEGADA', 'QTDE_VIAGENS_12M']
ESCALONADAS = ['TEMPO_VOO_ESC', 'ATRASO_CHEGADA_ESC', 'QTDE_VIAGENS_12M_ESC']

amostra = df.head(10).set_index('RESPONDENT_ID')

originais = amostra[VARIAVEIS]
escalonados = amostra[ESCALONADAS].round(4)

print('Dados originais')
display(originais)
print()
print('Dados escalonados')
display(escalonados)

Dados originais


,TEMPO_VOO,ATRASO_CHEGADA,QTDE_VIAGENS_12M
RESPONDENT_ID,,,
28211922,640.0,0,0.0
28211924,90.0,0,0.0
28211938,110.0,0,0.0
28211953,75.0,0,1.0
28211984,210.0,0,0.0
28211986,65.0,76,13.0
28211989,95.0,0,10.0
28211999,340.0,33,0.0
28212003,65.0,31,0.0



Dados escalonados


,TEMPO_VOO_ESC,ATRASO_CHEGADA_ESC,QTDE_VIAGENS_12M_ESC
RESPONDENT_ID,,,
28211922,2.0781,-0.1880,0.0000
28211924,-0.5625,-0.1880,0.0000
28211938,-0.4664,-0.1880,0.0000
28211953,-0.6345,-0.1880,0.0093
28211984,0.0137,-0.1880,0.0000
28211986,-0.6825,0.3689,0.1215
28211989,-0.5385,-0.1880,0.0935
28211999,0.6378,0.0538,0.0000
28212003,-0.6825,0.0392,0.0000


Duas verificações rápidas sobre a amostra, que o texto da A.1.6 comenta: quantos dos dez registros
têm atraso zero, e onde a coluna normalizada se posiciona no intervalo disponível.

In [5]:
pontuais = int((amostra['ATRASO_CHEGADA'] == 0).sum())
print(f'registros da amostra com atraso zero: {pontuais}, todos em {escalonados["ATRASO_CHEGADA_ESC"].min():.4f}')
print(
    f'QTDE_VIAGENS_12M escalonada na amostra: de {escalonados["QTDE_VIAGENS_12M_ESC"].min():.4f} '
    f'a {escalonados["QTDE_VIAGENS_12M_ESC"].max():.4f}'
)

registros da amostra com atraso zero: 6, todos em -0.1880
QTDE_VIAGENS_12M escalonada na amostra: de 0.0000 a 0.1215
